
# Exploratory Data Analysis – Curated Vehicle Dataset

- Compute image-level appearance statistics
- Aggregate scene-level statistics


## VS Code + hosted Colab setup

The notebook file stays local while the hosted Colab kernel reads the dataset from Google Drive. Rerun the next cell to refresh all paths; restarting the kernel is not required.


In [ ]:

import math
import os
import shutil
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from matplotlib.colors import rgb_to_hsv

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [ ]:
from google.colab import drive

DRIVE_MOUNT = Path("/content/drive")
if not (DRIVE_MOUNT / "MyDrive").is_dir():
    drive.mount(str(DRIVE_MOUNT))

DRIVE_PROJECT_ROOT = DRIVE_MOUNT / "MyDrive" / "ITU" / "3D" / "Thesis"
DATA_SEARCH_ROOT = DRIVE_PROJECT_ROOT / "data" / "COLMAP_curated"
matches = list(DATA_SEARCH_ROOT.rglob("curated_metadata.csv")) if DATA_SEARCH_ROOT.is_dir() else []
if len(matches) != 1:
    raise FileNotFoundError(
        f"Expected exactly one curated_metadata.csv under {DATA_SEARCH_ROOT}, found {len(matches)}."
    )

CSV_PATH = matches[0]
DATA_ROOT = CSV_PATH.parent
DEFAULT_IMAGE_ROOT = DATA_ROOT / "cars"
PLOT_DIR = DRIVE_PROJECT_ROOT / "plot" / "EDA"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Auto-detected Drive dataset at: {DATA_ROOT}")

# Optional: set this to the original raw image root if you want to resolve local files
# when `local_image_path` is not already present in the metadata.
RAW_IMAGE_ROOT = None

print("CSV_PATH:", CSV_PATH)
print("DEFAULT_IMAGE_ROOT:", DEFAULT_IMAGE_ROOT)
print("PLOT_DIR:", PLOT_DIR)


In [ ]:
df = pd.read_csv(CSV_PATH)
print("Rows:", len(df))
print("Columns:", list(df.columns))
display(df.head())


## 1. Dataset-level summary

In [ ]:

scene_summary = (
    df.groupby("CARID")
    .agg(
        n_images=("PICTUREID", "count"),
        n_viewclasses=("VIEWCLASS", "nunique"),
        MAKENAME=("MAKENAME", "first"),
        MODELNAME=("MODELNAME", "first"),
        VARIANTNAME=("VARIANTNAME", "first"),
    )
    .reset_index()
)

print("Number of scenes (CARIDs):", scene_summary["CARID"].nunique())
print("Images per scene:\n", scene_summary["n_images"].describe())
print("Viewclasses per scene:\n", scene_summary["n_viewclasses"].describe())
display(scene_summary)


In [ ]:
# Diversity across make / model / variant
variant_stats = (
    scene_summary
    .groupby(["MAKENAME", "MODELNAME", "VARIANTNAME"], dropna=False)
    .agg(
        car_count=("CARID", "nunique"),
        min_images=("n_images", "min"),
        median_images=("n_images", "median"),
        max_images=("n_images", "max"),
        min_viewclasses=("n_viewclasses", "min"),
        max_viewclasses=("n_viewclasses", "max"),
    )
    .sort_values(["car_count", "MAKENAME", "MODELNAME"], ascending=[False, True, True])
    .reset_index()
)

display(variant_stats)


In [ ]:

car_view_table = pd.pivot_table(
    df,
    index="CARID",
    columns="VIEWCLASS",
    values="PICTUREID",
    aggfunc="count",
    fill_value=0,
)

viewclass_counts = df["VIEWCLASS"].value_counts().sort_index()
print("Viewclass counts:")
display(viewclass_counts.to_frame("count"))

print("Images per CARID × VIEWCLASS:")
display(car_view_table)


## 2. Local image path resolution and scene visualization

In [ ]:

def build_image_lookup(image_root, exts=(".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff")):
    image_root = Path(image_root)
    lookup_by_stem = {}
    for p in image_root.rglob("*"):
        if p.is_file() and p.suffix.lower() in exts:
            lookup_by_stem[p.stem.strip()] = p
            # Exported files commonly look like FRONT_69502105.png.
            for token in p.stem.split("_"):
                if token.isdigit():
                    lookup_by_stem[token] = p
    return lookup_by_stem


def resolve_local_image_paths(df, default_image_root=DEFAULT_IMAGE_ROOT, raw_image_root=RAW_IMAGE_ROOT):
    out = df.copy()

    # Paths saved in the CSV may come from Windows and are not valid in Colab.
    # Always resolve them again against the mounted Drive folders.

    lookup = {}
    if default_image_root is not None and Path(default_image_root).exists():
        lookup.update(build_image_lookup(default_image_root))
    if raw_image_root is not None and Path(raw_image_root).exists():
        lookup.update(build_image_lookup(raw_image_root))

    def _resolve(row):
        car_dir = Path(default_image_root) / f"car_{row['CARID']}"
        filenames = []
        for col in ("exported_image_name", "standard_filename", "local_image_path"):
            value = row.get(col)
            if pd.notna(value):
                # replace handles filenames saved with Windows separators
                filenames.append(str(value).replace("\\", "/").rsplit("/", 1)[-1])

        for filename in filenames:
            for candidate in (car_dir / "images" / filename, car_dir / filename):
                if candidate.is_file():
                    return str(candidate)

        key = str(row["PICTUREID"]).strip()
        return str(lookup[key]) if key in lookup else None

    out["local_image_path"] = out.apply(_resolve, axis=1)
    print("Resolved rows:", out["local_image_path"].notna().sum(), "/", len(out))
    return out


df = resolve_local_image_paths(df)
display(df[[c for c in ["CARID", "PICTUREID", "VIEWCLASS", "local_image_path"] if c in df.columns]].head())


In [ ]:

def load_image_safely(path, bg_color=(255, 255, 255)):
    img = Image.open(path)
    if img.mode in ("RGBA", "LA"):
        bg = Image.new("RGB", img.size, bg_color)
        bg.paste(img, mask=img.getchannel("A"))
        return bg
    return img.convert("RGB")


def make_contact_sheet_from_paths(
    image_paths,
    thumb_size=(140, 100),
    max_cols=2,
    cell_bg="white",
    pad=10,
    crop=False,
):
    pics = []
    for path in image_paths:
        if path is None or pd.isna(path):
            continue
        path = Path(path)
        if not path.exists():
            continue
        img = load_image_safely(path)
        if crop:
            img = ImageOps.fit(img, thumb_size, method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))
        else:
            img.thumbnail(thumb_size, Image.Resampling.LANCZOS)
            canvas = Image.new("RGB", thumb_size, cell_bg)
            off_x = (thumb_size[0] - img.size[0]) // 2
            off_y = (thumb_size[1] - img.size[1]) // 2
            canvas.paste(img, (off_x, off_y))
            img = canvas
        pics.append(img)

    if len(pics) == 0:
        return Image.new("RGB", thumb_size, cell_bg)

    n = len(pics)
    cols = min(max_cols, n)
    rows = math.ceil(n / cols)
    sheet_w = cols * thumb_size[0] + (cols + 1) * pad
    sheet_h = rows * thumb_size[1] + (rows + 1) * pad
    sheet = Image.new("RGB", (sheet_w, sheet_h), cell_bg)

    for i, img in enumerate(pics):
        r = i // cols
        c = i % cols
        x = pad + c * (thumb_size[0] + pad)
        y = pad + r * (thumb_size[1] + pad)
        sheet.paste(img, (x, y))
    return sheet


def plot_car_view_grid(
    df,
    carids=None,
    path_col="local_image_path",
    thumb_size=(140, 100),
    max_cols_per_cell=2,
    crop=False,
):
    canonical_views = [
        "FRONT", "FRONT_LEFT", "FRONT_RIGHT", "SIDE_LEFT",
        "SIDE_RIGHT", "REAR_LEFT", "REAR_RIGHT", "REAR"
    ]

    work = df.copy()
    if carids is not None:
        work = work[work["CARID"].isin(carids)]
    carids = list(work["CARID"].drop_duplicates())

    if len(carids) == 0:
        print("No CARIDs selected.")
        return

    fig, axes = plt.subplots(len(carids), len(canonical_views), figsize=(2.4 * len(canonical_views), 2.2 * len(carids)))
    if len(carids) == 1:
        axes = np.array([axes])
    if len(canonical_views) == 1:
        axes = axes[:, None]

    for r, carid in enumerate(carids):
        car_df = work[work["CARID"] == carid]
        for c, view in enumerate(canonical_views):
            ax = axes[r, c]
            sub = car_df[car_df["VIEWCLASS"].astype(str).str.upper() == view]
            paths = sub[path_col].dropna().tolist() if path_col in sub.columns else []
            sheet = make_contact_sheet_from_paths(paths, thumb_size=thumb_size, max_cols=max_cols_per_cell, crop=crop)
            ax.imshow(sheet)
            ax.set_xticks([])
            ax.set_yticks([])
            if r == 0:
                ax.set_title(view, fontsize=9)
            if c == 0:
                ax.set_ylabel(str(carid), fontsize=9)

    plt.tight_layout()
    plt.show()


In [ ]:
example_carids = scene_summary["CARID"].tolist()
if len(example_carids) > 23:
    example_carids = example_carids[:]

plot_car_view_grid(df=df, carids=example_carids, path_col="local_image_path", thumb_size=(140, 100), max_cols_per_cell=2, crop=False)


## 3. Image-level statistics (including color and shininess proxies)

In [ ]:

def classify_dominant_color(arr):
    rgb = arr.astype(np.float32) / 255.0
    hsv = rgb_to_hsv(rgb)

    h = hsv[..., 0].ravel() * 360.0
    s = hsv[..., 1].ravel()
    v = hsv[..., 2].ravel()

    neutral_mask = s < 0.15
    if neutral_mask.mean() > 0.6:
        if v.mean() > 0.72:
            return "white"
        elif v.mean() < 0.28:
            return "black"
        else:
            return "gray"

    chromatic_mask = (s >= 0.15) & (v >= 0.15)
    if chromatic_mask.sum() == 0:
        return "gray"

    h = h[chromatic_mask]
    color_bins = {
        "red": ((h < 15) | (h >= 345)).sum(),
        "orange": ((h >= 15) & (h < 45)).sum(),
        "yellow": ((h >= 45) & (h < 70)).sum(),
        "green": ((h >= 70) & (h < 170)).sum(),
        "blue": ((h >= 170) & (h < 270)).sum(),
        "purple": ((h >= 270) & (h < 345)).sum(),
    }
    return max(color_bins, key=color_bins.get)


def compute_colorfulness(arr):
    arr = arr.astype(np.float32)
    rg = arr[:, :, 0] - arr[:, :, 1]
    yb = 0.5 * (arr[:, :, 0] + arr[:, :, 1]) - arr[:, :, 2]
    std_rg, std_yb = rg.std(), yb.std()
    mean_rg, mean_yb = rg.mean(), yb.mean()
    return float(np.sqrt(std_rg**2 + std_yb**2) + 0.3 * np.sqrt(mean_rg**2 + mean_yb**2))


def compute_shininess_metrics(arr):
    rgb = arr.astype(np.float32) / 255.0
    hsv = rgb_to_hsv(rgb)
    s = hsv[..., 1]
    v = hsv[..., 2]
    gray = arr.mean(axis=2)

    highlight_mask = (v > 0.85) & (s < 0.25)
    highlight_ratio = float(highlight_mask.mean())
    brightness_std = float(gray.std() / 255.0)
    p95 = float(np.percentile(v, 95))
    p99 = float(np.percentile(v, 99))
    highlight_peak = max(0.0, p99 - p95)

    shininess_score = 0.55 * highlight_ratio + 0.30 * brightness_std + 0.15 * highlight_peak
    return highlight_ratio, shininess_score


def compute_image_stats(img_path):
    img = Image.open(img_path).convert("RGB")
    arr = np.array(img).astype(np.float32)

    h, w = arr.shape[:2]
    gray = arr.mean(axis=2)
    gx = np.diff(gray, axis=1)
    gy = np.diff(gray, axis=0)
    sharpness = float(np.mean(np.abs(gx)) + np.mean(np.abs(gy)))

    dominant_color = classify_dominant_color(arr)
    colorfulness = compute_colorfulness(arr)
    highlight_ratio, shininess_score = compute_shininess_metrics(arr)

    return {
        "WIDTH_IMG": w,
        "HEIGHT_IMG": h,
        "ASPECT_RATIO": w / h if h > 0 else np.nan,
        "BRIGHTNESS_MEAN": float(gray.mean()),
        "BRIGHTNESS_STD": float(gray.std()),
        "R_MEAN": float(arr[:, :, 0].mean()),
        "G_MEAN": float(arr[:, :, 1].mean()),
        "B_MEAN": float(arr[:, :, 2].mean()),
        "SHARPNESS": sharpness,
        "DOMINANT_COLOR": dominant_color,
        "COLORFULNESS": colorfulness,
        "HIGHLIGHT_RATIO": highlight_ratio,
        "SHININESS_SCORE": shininess_score,
    }


def add_image_stats_to_df(df, path_col="local_image_path"):
    out = df.copy()
    stats_rows = []

    for _, row in out.iterrows():
        rec = {"PICTUREID": row["PICTUREID"]}
        path = row.get(path_col, None)
        if path is None or pd.isna(path) or not Path(path).exists():
            stats_rows.append(rec)
            continue
        try:
            rec.update(compute_image_stats(path))
        except Exception as e:
            print(f"Failed on {row['PICTUREID']}: {e}")
        stats_rows.append(rec)

    stats_df = pd.DataFrame(stats_rows)
    out = out.merge(stats_df, on="PICTUREID", how="left")
    return out


In [ ]:

df_stats = add_image_stats_to_df(df, path_col="local_image_path")
print("Added columns:")
print([c for c in ["WIDTH_IMG", "HEIGHT_IMG", "ASPECT_RATIO", "BRIGHTNESS_MEAN", "BRIGHTNESS_STD", "SHARPNESS", "DOMINANT_COLOR", "COLORFULNESS", "HIGHLIGHT_RATIO", "SHININESS_SCORE"] if c in df_stats.columns])
display(df_stats)


## 4. Scene-level statistics

In [ ]:

def compute_scene_stats(df):
    named_aggs = {
        "n_images": ("PICTUREID", "count"),
        "n_viewclasses": ("VIEWCLASS", "nunique"),
    }

    base_optional = {
        "WIDTH_IMG": ["width_mean", "width_std"],
        "HEIGHT_IMG": ["height_mean", "height_std"],
        "ASPECT_RATIO": ["aspect_ratio_mean", "aspect_ratio_std"],
        "BRIGHTNESS_MEAN": ["brightness_mean", "brightness_std_between_imgs"],
        "BRIGHTNESS_STD": ["contrast_mean", "contrast_std_between_imgs"],
        "R_MEAN": ["r_mean"],
        "G_MEAN": ["g_mean"],
        "B_MEAN": ["b_mean"],
        "SHARPNESS": ["sharpness_mean", "sharpness_std"],
        "COLORFULNESS": ["colorfulness_mean", "colorfulness_std"],
        "HIGHLIGHT_RATIO": ["highlight_ratio_mean", "highlight_ratio_std"],
        "SHININESS_SCORE": ["shininess_mean", "shininess_std"],
    }

    reducers = {
        1: ["mean"],
        2: ["mean", "std"],
    }

    for col, out_names in base_optional.items():
        if col in df.columns:
            ops = reducers[len(out_names)]
            for out_name, op in zip(out_names, ops):
                named_aggs[out_name] = (col, op)

    scene_stats = df.groupby("CARID").agg(**named_aggs).reset_index()

    if {"width_mean", "width_std", "height_mean", "height_std"}.issubset(scene_stats.columns):
        scene_stats["resolution_cv"] = (
            (scene_stats["width_std"] / scene_stats["width_mean"]).fillna(0) +
            (scene_stats["height_std"] / scene_stats["height_mean"]).fillna(0)
        ) / 2
    else:
        scene_stats["resolution_cv"] = np.nan

    if "DOMINANT_COLOR" in df.columns:
        scene_color = (
            df.groupby("CARID")["DOMINANT_COLOR"]
            .agg(lambda x: x.mode().iat[0] if not x.mode().empty else np.nan)
            .rename("dominant_color")
            .reset_index()
        )
        scene_stats = scene_stats.merge(scene_color, on="CARID", how="left")

    for meta_col in ["MAKENAME", "MODELNAME", "VARIANTNAME"]:
        if meta_col in df.columns:
            meta = df.groupby("CARID")[meta_col].first().rename(meta_col).reset_index()
            scene_stats = scene_stats.merge(meta, on="CARID", how="left")

    return scene_stats


In [ ]:

scene_stats = compute_scene_stats(df_stats)
print("Scene-level columns:", list(scene_stats.columns))
display(scene_stats)


## 5. Thresholds and reconstruction readiness

In [ ]:

THRESHOLDS = {
    "min_images": 6,
    "min_viewclasses": 6,
    "max_resolution_cv": 0.05,
    "max_brightness_std_between_imgs": 25.0,
    "min_sharpness_mean": 8.0,
    "max_shininess_mean": 0.12,
    "max_highlight_ratio_mean": 0.08,
}

THRESHOLDS


In [ ]:

def add_recon_flags(scene_stats, thresholds):
    recon_ready = scene_stats.copy()

    recon_ready["enough_images"] = recon_ready["n_images"] >= thresholds["min_images"]
    recon_ready["good_view_coverage"] = recon_ready["n_viewclasses"] >= thresholds["min_viewclasses"]
    recon_ready["resolution_consistent"] = recon_ready["resolution_cv"] <= thresholds["max_resolution_cv"]

    if "brightness_std_between_imgs" in recon_ready.columns:
        recon_ready["brightness_consistent"] = recon_ready["brightness_std_between_imgs"] <= thresholds["max_brightness_std_between_imgs"]
    else:
        recon_ready["brightness_consistent"] = True

    if "sharpness_mean" in recon_ready.columns:
        recon_ready["sharp_enough"] = recon_ready["sharpness_mean"] >= thresholds["min_sharpness_mean"]
    else:
        recon_ready["sharp_enough"] = True

    if "shininess_mean" in recon_ready.columns:
        recon_ready["not_too_shiny"] = recon_ready["shininess_mean"] <= thresholds["max_shininess_mean"]
    else:
        recon_ready["not_too_shiny"] = True

    if "highlight_ratio_mean" in recon_ready.columns:
        recon_ready["limited_highlights"] = recon_ready["highlight_ratio_mean"] <= thresholds["max_highlight_ratio_mean"]
    else:
        recon_ready["limited_highlights"] = True

    flag_cols = [
        "enough_images",
        "good_view_coverage",
        "resolution_consistent",
        "brightness_consistent",
        "sharp_enough",
        "not_too_shiny",
        "limited_highlights",
    ]

    recon_ready["READY_FOR_RECON"] = recon_ready[flag_cols].all(axis=1)
    return recon_ready


recon_ready = add_recon_flags(scene_stats, THRESHOLDS)
display(recon_ready)


## 6. Diagnostic plots after thresholds (saved to `plot/EDA`)

In [ ]:

def save_fig(name):
    out = PLOT_DIR / f"{name}.png"
    plt.tight_layout()
    plt.savefig(out, dpi=200, bbox_inches="tight")
    print("Saved:", out)


def plot_hist_with_threshold(series, threshold=None, threshold_type="max", xlabel="", title="", filename="plot"):
    plt.figure(figsize=(7, 4))
    series = pd.Series(series).dropna()
    plt.hist(series, bins=min(30, max(5, int(np.sqrt(len(series))))), color="black")
    if threshold is not None:
        plt.axvline(threshold, linestyle="--", linewidth=1.5)
        label = f"threshold = {threshold} ({threshold_type})"
        ymax = plt.gca().get_ylim()[1]
        plt.text(threshold, ymax * 0.9, label, rotation=90, va="top", ha="right")
    plt.xlabel(xlabel)
    plt.ylabel("Count")
    plt.title(title)
    save_fig(filename)
    plt.show()


In [ ]:

plot_hist_with_threshold(
    recon_ready["n_images"],
    threshold=THRESHOLDS["min_images"],
    threshold_type="min",
    xlabel="Images per scene",
    title="Images per scene",
    filename="images_per_scene_threshold",
)

plot_hist_with_threshold(
    recon_ready["n_viewclasses"],
    threshold=THRESHOLDS["min_viewclasses"],
    threshold_type="min",
    xlabel="Unique viewclasses per scene",
    title="View coverage per scene",
    filename="viewclasses_per_scene_threshold",
)

plot_hist_with_threshold(
    recon_ready["resolution_cv"],
    threshold=THRESHOLDS["max_resolution_cv"],
    threshold_type="max",
    xlabel="Resolution coefficient of variation",
    title="Resolution consistency per scene",
    filename="resolution_cv_threshold",
)

if "brightness_std_between_imgs" in recon_ready.columns:
    plot_hist_with_threshold(
        recon_ready["brightness_std_between_imgs"],
        threshold=THRESHOLDS["max_brightness_std_between_imgs"],
        threshold_type="max",
        xlabel="Brightness std within scene",
        title="Brightness consistency per scene",
        filename="brightness_std_threshold",
    )

if "sharpness_mean" in recon_ready.columns:
    plot_hist_with_threshold(
        recon_ready["sharpness_mean"],
        threshold=THRESHOLDS["min_sharpness_mean"],
        threshold_type="min",
        xlabel="Mean sharpness per scene",
        title="Sharpness per scene",
        filename="sharpness_threshold",
    )

if "shininess_mean" in recon_ready.columns:
    plot_hist_with_threshold(
        recon_ready["shininess_mean"],
        threshold=THRESHOLDS["max_shininess_mean"],
        threshold_type="max",
        xlabel="Mean shininess score per scene",
        title="Shininess per scene",
        filename="shininess_threshold",
    )

if "highlight_ratio_mean" in recon_ready.columns:
    plot_hist_with_threshold(
        recon_ready["highlight_ratio_mean"],
        threshold=THRESHOLDS["max_highlight_ratio_mean"],
        threshold_type="max",
        xlabel="Mean highlight ratio per scene",
        title="Highlight ratio per scene",
        filename="highlight_ratio_threshold",
    )


## 7. Color and shininess summaries

In [ ]:

if "DOMINANT_COLOR" in df_stats.columns:
    print("Image-level dominant color counts:")
    display(df_stats["DOMINANT_COLOR"].value_counts(dropna=False).to_frame("count"))

if "dominant_color" in scene_stats.columns:
    print("Scene-level dominant color counts:")
    display(scene_stats["dominant_color"].value_counts(dropna=False).to_frame("count"))

if {"CARID", "dominant_color", "shininess_mean", "highlight_ratio_mean", "n_images"}.issubset(scene_stats.columns):
    display(
        scene_stats.sort_values("shininess_mean", ascending=False)[
            ["CARID", "dominant_color", "shininess_mean", "highlight_ratio_mean", "n_images"]
        ].head(10)
    )


In [ ]:
for carid, group in df_stats.groupby("CARID"):
    widths = group["WIDTH_IMG"].values
    heights = group["HEIGHT_IMG"].values

    print(f"\nCARID: {carid}")
    print("Widths:", widths)
    print("Heights:", heights)
    print("Width CV:", widths.std() / widths.mean())
    print("Height CV:", heights.std() / heights.mean())

## 8. Problem scenes and failure reasons

In [ ]:

flag_cols = [
    "enough_images",
    "good_view_coverage",
    "resolution_consistent",
    "brightness_consistent",
    "sharp_enough",
    "not_too_shiny",
    "limited_highlights",
]

problem_scenes = recon_ready.loc[~recon_ready["READY_FOR_RECON"]].copy()
problem_scenes["failed_checks"] = problem_scenes[flag_cols].apply(
    lambda row: [col for col, ok in row.items() if not ok], axis=1
)

display(problem_scenes[["CARID", "MAKENAME", "MODELNAME", "VARIANTNAME", "n_images", "n_viewclasses", "READY_FOR_RECON", "failed_checks"] + [c for c in ["dominant_color", "shininess_mean"] if c in problem_scenes.columns]])


## 9. Optional export: build curated COLMAP-ready dataset

In [ ]:
RUN_CURATION = False  # Set True only when you intentionally want to rebuild the Drive export.
CURATED_ROOT = DATA_ROOT
CARS_DIR = CURATED_ROOT / "cars"
CURATED_CSV = CURATED_ROOT / "curated_metadata.csv"
MIN_VIEWS_PER_CAR = 6
OVERWRITE = False

CANONICAL_VIEWS = [
    "FRONT", "FRONT_LEFT", "SIDE_LEFT", "REAR_LEFT",
    "REAR", "REAR_RIGHT", "SIDE_RIGHT", "FRONT_RIGHT",
]

KEEP_COLS = [
    "CARID",
    "PICTUREID",
    "VIEWCLASS",
    "local_image_path",
    "WIDTH_IMG",
    "HEIGHT_IMG",
    "ASPECT_RATIO",
    "BRIGHTNESS_MEAN",
    "BRIGHTNESS_STD",
    "R_MEAN",
    "G_MEAN",
    "B_MEAN",
    "SHARPNESS",
    "DOMINANT_COLOR",
    "COLORFULNESS",
    "HIGHLIGHT_RATIO",
    "SHININESS_SCORE",
]


In [ ]:
def norm_viewclass(x):
    if pd.isna(x):
        return None
    x = str(x).strip().upper()
    aliases = {
        "FRONTLEFT": "FRONT_LEFT",
        "FRONTRIGHT": "FRONT_RIGHT",
        "REARLEFT": "REAR_LEFT",
        "REARRIGHT": "REAR_RIGHT",
        "LEFT": "SIDE_LEFT",
        "RIGHT": "SIDE_RIGHT",
        "SIDELEFT": "SIDE_LEFT",
        "SIDERIGHT": "SIDE_RIGHT",
    }
    return aliases.get(x, x)


def choose_best_row_per_view(group):
    g = group.copy()

    width_col = "WIDTH_IMG" if "WIDTH_IMG" in g.columns else "WIDTH"
    height_col = "HEIGHT_IMG" if "HEIGHT_IMG" in g.columns else "HEIGHT"

    if width_col in g.columns and height_col in g.columns:
        g["_area"] = (
            pd.to_numeric(g[width_col], errors="coerce").fillna(0) *
            pd.to_numeric(g[height_col], errors="coerce").fillna(0)
        )
        g = g.sort_values("_area", ascending=False)

    return g.iloc[0]


def build_pairs_for_views(views_in_car):
    present = [v for v in CANONICAL_VIEWS if v in views_in_car]
    pairs = []

    for i in range(len(present)):
        v1 = present[i]
        v2 = present[(i + 1) % len(present)]
        if v1 != v2:
            pairs.append((v1, v2))

    return pairs


def export_curated_dataset(df_input):
    curated = df_input.copy()

    curated["VIEWCLASS"] = curated["VIEWCLASS"].map(norm_viewclass)
    curated = curated[curated["VIEWCLASS"].isin(CANONICAL_VIEWS)].copy()
    curated = curated[curated["local_image_path"].notna()].copy()

    if OVERWRITE and CURATED_ROOT.exists():
        shutil.rmtree(CURATED_ROOT)

    CARS_DIR.mkdir(parents=True, exist_ok=True)
    exported_rows = []

    for carid, g in curated.groupby("CARID"):
        selected = (
            g.groupby("VIEWCLASS", group_keys=False)
            .apply(choose_best_row_per_view)
            .reset_index(drop=True)
        )

        if selected["VIEWCLASS"].nunique() < MIN_VIEWS_PER_CAR:
            continue

        car_dir = CARS_DIR / f"car_{carid}"
        img_dir = car_dir / "images"
        img_dir.mkdir(parents=True, exist_ok=True)

        view_to_name = {}

        for _, row in selected.iterrows():
            src = Path(row["local_image_path"])
            ext = src.suffix.lower() if src.suffix else ".png"
            dst_name = f"{row['VIEWCLASS']}_{row['PICTUREID']}{ext}"
            dst = img_dir / dst_name

            shutil.copy2(src, dst)

            row_out = row.to_dict()
            row_out["exported_image_name"] = dst_name
            row_out["exported_image_path"] = str(dst)

            exported_rows.append(row_out)
            view_to_name[row["VIEWCLASS"]] = dst_name

        pairs = build_pairs_for_views(set(selected["VIEWCLASS"]))

        with open(car_dir / "pairs.txt", "w", encoding="utf-8") as f:
            for v1, v2 in pairs:
                f.write(f"{view_to_name[v1]} {view_to_name[v2]}\n")

    exported_df = pd.DataFrame(exported_rows)

    front_cols = [
        c for c in [
            "CARID",
            "PICTUREID",
            "VIEWCLASS",
            "local_image_path",
            "exported_image_name",
            "exported_image_path",
        ]
        if c in exported_df.columns
    ]
    other_cols = [c for c in exported_df.columns if c not in front_cols]
    exported_df = exported_df[front_cols + other_cols].copy()

    exported_df.to_csv(CURATED_CSV, index=False)

    print("Exported rows:", len(exported_df))
    print("Saved curated metadata to:", CURATED_CSV)
    print("Saved columns:")
    print(exported_df.columns.tolist())

    return exported_df


if RUN_CURATION:
    curated_df = export_curated_dataset(df_stats)
    display(curated_df.head())
else:
    print("RUN_CURATION is False. Set it to True to export the curated dataset.")